# Welcome to the 4DVar tutorial
***

In this tutorial we will explore the basic features of 4DVar, using the quasi-geostrophic model. **Be sure to run 0.qg_tutorial_start.ipynb before this tutorial.** **It is also recommended that you run the 3DVar tutorial first**. The model mimics the evolution of large-scale features, such as high and low pressure cells in the atmosphere. In the following we first provide a short description of the 4DVar problem and of the quasi-geostrophic model and what it represents, followed by the data-assimilation experiments.

## Introduction to 4DVar

4DVar extends 3DVar by assimilating observations distributed across a time window $[0,T]$, rather than treating every observation as valid at a single analysis time. We start with a background (first-guess) state $x_b$ at the beginning of the window and a set of observations $y_t$ collected at times $t = 0, 1, \ldots, T$, each with observation-error covariance $R_t$. The goal is to find the state $x_0$ at the start of the window that is both close to $x_b$ and, when run forward through the forecast model $M_{0 \rightarrow t}$, agrees with the observations at their respective times.

The cost function is similar to the 3Dvar form, 
<br><br>
<center>$J(x_0) = (x_0 - x_b)^T B^{-1} (x_0 - x_b) + \sum_{t=0}^{T} \left(y_t - H_t M_{0 \rightarrow t}(x_0)\right)^T R_t^{-1} \left(y_t - H_t M_{0 \rightarrow t}(x_0)\right)$</center>
<br>
but with two key changes from 3DVar: (i) each observation $y_t$ is compared against a <i>model forecast</i> $M_{0 \rightarrow t}(x_0)$ rather than against $x_0$ directly, and (ii) the observation operator $H_t$ is applied to that forecast at observation time $t$. The control variable is the state $x_0$ at $t=0$; once it is found, the analysis at any later time is simply $M_{0 \rightarrow t}(x_0)$, which means the analysis trajectory is <i>dynamically consistent</i> with the model.<br><br>

<u>**Main benefit over 3DVar.**</u> Because the analysis is solved at $t=0$ but observations are spread throughout the window, 4DVar implicitly *propagates the static background-error covariance $B$ forward in time* through the **tangent-linear model (TLM) $M_t$**, producing a flow-dependent implied covariance $M_t B M_t^T$ at each observation time. Information from each observation is then carried back to the start of the window using the **adjoint of the TLM**, $M_t^T$. In effect, the analysis at $t=0$ "sees" a covariance that has been shaped by the dynamics of the flow across the window, allowing observations to spread in physically meaningful, flow-dependent ways and giving 4DVar a truly four-dimensional structure even though $B$ itself is static. 3DVar, which uses $B$ at a single time, cannot do this.

<u>**Solving the problem in practice.**</u> Because $M_{0 \rightarrow t}$ and $H_t$ are generally nonlinear, the cost function is non-quadratic and is minimized iteratively using **incremental 4DVar** (a Gauss-Newton scheme). The minimization is split into two nested loops:

- An **outer loop** runs the full nonlinear model from a current guess $x_0^{(i)}$ over the window. This trajectory is used to linearize the model and observation operators, producing the TLM $M_t$ and the linearized $H_t$ valid along that trajectory, as well as the innovations $d_t = y_t - H_t M_{0 \rightarrow t}(x_0^{(i)})$ at each observation time.
- An **inner loop** then minimizes a quadratic cost function for an increment $\delta x_0$ at $t=0$. The iterate is updated as $x_0^{(i+1)} = x_0^{(i)} + \delta x_0$, and the outer loop is repeated until convergence.

Each inner iteration looks like: a **forward TLM integration** of $\delta x_0$ through the window to evaluate residuals against the innovations $d_t$, followed by a **backward adjoint integration** that accumulates those residuals into the gradient with respect to $\delta x_0$ at $t=0$. A linear solver (typically conjugate-gradient or Lanczos) uses this gradient to update $\delta x_0$.

Because the system is high-dimensional, a **preconditioner** is applied to the inner loop to make the minimization well-conditioned. In JEDI the standard choice is to precondition with $B$ itself (a control-variable transform), so that the effective minimization is over a "whitened" variable in which $B$ acts as the identity.

**Further reading.** For students who want to delve further into 4dvar theory, two good references are: 
- F. Bouttier and P. Courtier (1999), [*Data Assimilation Concepts and Methods*](https://www.ecmwf.int/sites/default/files/elibrary/2002/16928-data-assimilation-concepts-and-methods.pdf).
- P. Courtier, J.-N. Thépaut, and A. Hollingsworth (1994), [*A strategy for operational implementation of 4D-Var, using an incremental approach*](https://rmets.onlinelibrary.wiley.com/doi/10.1002/qj.49712051912), Quarterly Journal of the Royal Meteorological Society, 120, 1367-1387, doi:10.1002/qj.49712051912 — the foundational paper for incremental 4DVar, the algorithm used here.

With this picture in hand, we can dive into the experiments with JEDI.

## Environment Setup

For convenience, you _need_ to export a few environment variables:

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

## Overview of the 4Dvar experiments

In the following you will conduct several experiments. Most will be experiments in which you will assimilate one observation. This will allow you to better understand the JEDI system, but you will especially appreciate how 4DVar works and how the position of the observation in the assimilation window is important.

| Description | Background yaml file   | Observation yaml file | Data assimilation file |
| --- | --- | --- | --- |
| exp_begin | genenspert_B_default.yaml | makeobs4d_oneobs_begin.yaml | 4dvar_oneobs_begin.yaml
| exp_end | genenspert_B_default.yaml | makeobs4d_oneobs_end.yaml | 4dvar_oneobs_end.yaml
| exp_mult_obs | genenspert_B_default.yaml | makeobs4d_mult_obs.yaml | 4dvar_mult_obs.yaml


Our last experiment with 4DVar will be a more realistic case with 100 randomly positioned observations.

# Experiment 1: 4dvar with one observation at the beginning of the window
***

### Step 1: Truth

The truth is the same as in the 3dvar, there is no need to generate it again!

In [ ]:
ls $JEDI_EDU/qgstart/output/truth/ 

Before we move on, let us plot the truth over the 24-h period that covers the assimilation window for 4dvar here. It will be useful to get a sense of the flow in time when evaluating the 4dvar results.

In [ ]:
cd $JEDI_EDU/plots_scripts
for fcsthr in P15D P15DT6H P15DT12H P15DT18H P16D; do
    python ./plot.py qg fields $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.${fcsthr}.nc \
            --plotwind --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/truth_${fcsthr}
done

You will see five images plotted every 6 hours between forecast days 15-16, which is the 34-h time window for which we will perform 4dvar experiments. Note how the flow mimics atmospheric flow - some areas waves progress from west to east, others are stationary like a blocking high, etc. Thes eflow characteristics are important considerations for evaluating how 4dvar produces increments from observations at different times in the DA window. 

<center><img src="images/truth_P15-16.gif" width="700"/></center>

### Step 2: Background state

We are using the same backgrounds as in the 3dvar run, but we will use a background 24hrs earlier (because the 4dvar window is 24hrs long). This will allow us to compare results between the 3dvar and 4dvar runs: the analyses will be valid at the same time.

In [ ]:
cp $JEDI_EDU/qgstart/output/bg/* $JEDI_EDU/qg4Dvar/output/exp_begin/bg
ls $JEDI_EDU/qg4Dvar/output/exp_begin/bg

We can create plots for the background error and look at the error for the stream function at the beginning of our DA window. Note in these experiments, the DA window is 24-h starting from 2009-12-30 00:00:00Z and ending at 2009-12-31 00:00:00Z:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_begin/bg/bkgd.fc.2009-12-30T00\:00\:00Z.PT0S.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P15D.nc \
        --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/bkgd_error --title "Background error - beginning of window"


### Step 3: Generate observations

Our assimilation window is from 2009-12-30T00:00:00 to 2009-12-31T00:00:00, and we will generate an observation just one hour after the beginning of the window.

Look at the following yaml code from `makeobs4d_oneobs_begin.yaml`:
```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
initial condition:
  date: 2009-12-30T00:00:00Z
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P15D.nc
model:
  name: QG
  tstep: PT6H
forecast length: PT24H
window begin: 2009-12-30T00:00:00Z
window length: PT24H
observations:
  observers:
  - obs operator:
      obs type: Stream              # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qg4Dvar/output/exp_begin/obs/truth.1obs.begin.nc
      obs type: Stream
      generate:
        begin: PT1H                 # The observation is generated one hour after the beginning of the window
        nval: 1
        obs locations:
          lon: [-125]                 # List of longitudes for generated observations (only one here)
          lat: [15]                 # List of latitudes for generated observations (only one here)
          z: [7000.0]               # List of heights for generated observations (only one here)
        obs error: 4.0e6            # Observation error standard deviation, in m^2/s
        obs period: PT24H
make obs: true                      # Generate the observations
obs perturbations: true             # Add random  measurements to the observations
```

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx.x ./qg4Dvar/yamls/makeobs4d_oneobs_begin.yaml

You can look into the observation file you just created and verify that there is only one location:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qg4Dvar/output/exp_begin/obs/truth.1obs.begin.nc \
        --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/qg_oneobs_begin
cat $JEDI_EDU/qg4Dvar/output/exp_begin/plots/qg_oneobs_begin.txt

### Step 4: Run the 4DVar!

Observe the contents of the 4dvar yaml file `qg4Dvar/yamls/4dvar_oneobs_begin.yaml`. You will find many items are similar to the 3dvar yaml file you just worked with:

```yaml
cost function:
  cost type: 4D-Var                        # The cost function used is now 4d-var
  window begin: 2009-12-30T00:00:00Z       # The beginning of the window is on the 30th at midnight
  window length: PT24H                     # The window is 24 hours long
  analysis variables: [x]
  background:
    date: 2009-12-30T00:00:00Z
    filename: qg4Dvar/output/exp_begin/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc
  background error:
    covariance model: QgError
    horizontal_length_scale: 2.2e6
    maximum_condition_number: 1.0e6
    standard_deviation: 1.8e7
    vertical_length_scale: 15000.0
  observations:
    observers:
    - obs operator:
        obs type: Stream
      obs space:
        obsdatain:
          engine:
            obsfile: qg4Dvar/output/exp_begin/obs/truth.1obs.begin.nc
        obsdataout:
          engine:
            obsfile: qg4Dvar/output/exp_begin/obs/4dvar.1obs.begin.nc
        obs type: Stream
      obs error:
        covariance model: diagonal
  geometry:
    nx: 40
    ny: 20
    depths: [4000.0, 6000.0]
  model:                                   # We need to specify the model that will be run
    name: QG
    tstep: PT1H
variational:
  minimizer:
    algorithm: DRPLanczos
  iterations:
  - ninner: 10
    gradient norm reduction: 1.0e-10
    geometry:
      nx: 40
      ny: 20
      depths: [4000.0, 6000.0]
    linear model:
      name: QgTLM
      trajectory:
        tstep: PT1H
      tstep: PT1H
      variable change: Identity
    diagnostics:
      departures: ombg
    test: on
  - ninner: 10
    gradient norm reduction: 1.0e-10
    geometry:
      nx: 40
      ny: 20
      depths: [4000.0, 6000.0]
    linear model:
      name: QgTLM
      trajectory:
        tstep: PT1H
      tstep: PT1H
      variable change: Identity
    diagnostics:
      departures: ombg
    test: on
final:
  diagnostics:
    departures: oman
  prints:
    frequency: PT1H
output:
  datadir: qg4Dvar/output/exp_begin/da/
  exp: 4dvar.1obs.begin
  first: PT0S
  frequency: PT6H
  type: an
```

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg4Dvar/yamls/4dvar_oneobs_begin.yaml

**You can now look at the results.**<br>
Let's plot the increment at the beginning of the window:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_begin/da/4dvar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_begin/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_begin/obs/4dvar.1obs.begin.nc \
        --plotwind --fieldmax 3e7 --scalemax \
        --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/4dvar_inc_begin --title "Increment 00H: obs at beginning of window"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_begin/da/4dvar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_begin/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_begin/obs/4dvar.1obs.begin.nc \
        --plotwind --fieldmax 3e7 --scalemax \
        --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/4dvar_inc_mid --title "Increment 12H: obs at beginning of window"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_begin/da/4dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_begin/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_begin/obs/4dvar.1obs.begin.nc \
        --plotwind --fieldmax 3e7 --scalemax \
        --output $JEDI_EDU/qg4Dvar/output/exp_begin/plots/4dvar_inc_end --title "Increment 24H: obs at beginning of window"


<center><img src="images/4dvar_inc_exp_begin.gif" width="700"/></center>

Examine the increments above and think about how to answer the following questions:

- What time are the increments most centralized around the observation location?
- The initial increments at 00H appear similar to 3DVar analysis increments - is this expected? Why or why not? (Hint: think about how much time the model has had to act on the increment.)
- How do the increments propagate in time? Is it dynamically-consistent with the evolution of the model in that location? (refer to the truth plots above!)
- Are there other dynamical features besides simple advection to point out? Look for changes in the increment's shape, size, or intensity as it evolves.

# Experiment 2: 4dvar with one observation at the end of the window
***

### Step 1: Truth

The truth stays the same

### Step 2:Background state

We can copy the backgrounds from the first experiment:

In [ ]:
cp $JEDI_EDU/qg4Dvar/output/exp_begin/bg/* $JEDI_EDU/qg4Dvar/output/exp_end/bg
ls $JEDI_EDU/qg4Dvar/output/exp_end/bg

### Step 3: Generate the observations

We need to generate one observation at the end of the window. the yaml file `makeobs4d_oneobs_end.yaml` has been provided. Note in addition to generating at the end (`begin: PT24H`), we have shifted the observation to the east (`lon: [-100]`) to follow the local maxima in streamfunction at the end of the window and see how increments are propagated backward in time via the adjoint of the model in 4dvar. 

In [ ]:
diff $JEDI_EDU/qg4Dvar/yamls/makeobs4d_oneobs_begin.yaml $JEDI_EDU/qg4Dvar/yamls/makeobs4d_oneobs_end.yaml


You can now generate the observations with:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx.x ./qg4Dvar/yamls/makeobs4d_oneobs_end.yaml
ls ./qg4Dvar/output/exp_end/obs

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qg4Dvar/output/exp_end/obs/truth.1obs.end.nc \
        --output $JEDI_EDU/qg4Dvar/output/exp_end/plots/qg_oneobs_end
cat $JEDI_EDU/qg4Dvar/output/exp_end/plots/qg_oneobs_end.txt

### Step 4: Run the 4dvar!


In [ ]:
diff $JEDI_EDU/qg4Dvar/yamls/4dvar_oneobs_begin.yaml $JEDI_EDU/qg4Dvar/yamls/4dvar_oneobs_end.yaml


First examine the differences in the yaml files above from the exp_begin experiment. We provide the 4dvar with our new `truth.1obs.end.nc` observation file, and change directories for inputs/outputs to `exp_end`

Then, run the 4dvar:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg4Dvar/yamls/4dvar_oneobs_end.yaml

We can look at the increment at the beginning, middle and end of the window: when do you expect it will best fit the observation?

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_end/da/4dvar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_end/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_end/obs/truth.1obs.end.nc \
        --plotwind --fieldmax 3e7 --scalemax\
        --output $JEDI_EDU/qg4Dvar/output/exp_end/plots/4dvar_inc_begin --title "Increment 00H: obs at end of window"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_end/da/4dvar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_end/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_end/obs/truth.1obs.end.nc \
        --plotwind --fieldmax 3e7 --scalemax \
        --output $JEDI_EDU/qg4Dvar/output/exp_end/plots/4dvar_inc_mid --title "Increment 12H: obs at end of window"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_end/da/4dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_end/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qg4Dvar/output/exp_end/obs/4dvar.1obs.end.nc \
        --plotwind --fieldmax 3e7 --scalemax \
        --output $JEDI_EDU/qg4Dvar/output/exp_end/plots/4dvar_inc_end --title "Increment 24H: obs at end of window"


<center><img src="images/4dvar_inc_exp_end.gif" width="700"/></center>

Examine these increments (plotted going from 24H backwards to 00H). Compare to the obs-at-the-beginning experiment and try to answer these questions:

- What time are the increments most centralized around the observation location?
- Do the increments at the time closest to observation time (24H) look like 3D-Var increments? Why is the 24H pattern in `exp_end` different from the 00H pattern in `exp_begin`, even though both are plotted at the obs valid time?
- How do the increments propagate backward in time towards 00H?
- How does it compare with the exp_begin experiment? Are there differences in the patterns?

Note we can think of the increments at 00H as a precursor pattern - the initial condition increment needed for the (linearized) model to focus, deform, and amplify the increments to the compacted area in the 24H figure. In a sense, we have backed out a weaker, broader scale increment at 00H for exp_end - similar to broad corrections made at 00H for the obs-at-the-beginning experiment. 

We can certainly point out important differences for thr 24H plot - for exp_end there is a northward elongation of positive streamfunction corrections that is not present in exp_begin at this time. This is because the dynamics in this region preferentially elongate features in certain directions, and that deformation acts over the full 24H window in exp_end (whereas exp_begin has the increment at its 'fresh' state at 00H, before the flow has acted on it). However, the evolution of the maximum increments - from a weak, broader area at 00H to a strong, spatially smaller area at 24H - are similar for both experiments, demonstrating for us how 4DVar can capture dynamically-consistent corrections both forward (via TLM) and backward (via adjoint) in time. 

To summarize: When the observation is at the beginning of the window, the 4D-Var increment at $t_0$ looks 3DVar-like, and the dynamics carry it forward via the linearized model in the 4dvar (propagating the error covariance matrix forward in time). When the observation is at the end of the window, the increment at $t_0$ is a precursor pattern that the dynamics will sharpen into the obs. In both cases, each analysis is dynamically consistent with the model.

# Experiment 3: 4dvar with multiple observation scattered through the window
***

The previous experiments gave us clean pictures of how 4D-Var spreads information from a single observation through the assimilation window: forward via the tangent linear model, backward via the adjoint. 

In this experiment, we'll assimilate 150 observations scattered across the 24-hour window — 50 observations every 3 hours — at random locations. This is still highly idealized compared to operational systems, but it's much closer to a realistic flow of information into the analysis.

### Step 1: Truth

The truth stays the same

### Step 2: Background state

We can copy the backgrounds from the first experiment:

In [ ]:
cp $JEDI_EDU/qg4Dvar/output/exp_begin/bg/* $JEDI_EDU/qg4Dvar/output/exp_mult/bg
ls $JEDI_EDU/qg4Dvar/output/exp_mult/bg

### Step 3: Generate the observations

We now move to a more realistic case with observations every 3 hours at 50 random locations. The yaml file used for this experiment is `makeobs4d_mult_obs.yaml`:

```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
initial condition:
  date: 2009-12-30T00:00:00Z
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P15D.nc
model:
  name: QG
  tstep: PT3H
forecast length: PT24H
window begin: 2009-12-30T00:00:00Z
window length: PT24H
observations:
  observers:
  - obs operator:
      obs type: Stream
    obs space:
      obsdataout:
        engine:
          obsfile: qg4Dvar/output/exp_mult/obs/truth.multobs.nc
      obs type: Stream
      generate:
        begin: PT1H
        nval: 1
        obs density: 50
        obs error: 4.0e6
        obs period: PT3H
make obs: true
obs perturbations: true
```

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx.x ./qg4Dvar/yamls/makeobs4d_mult_obs.yaml

### Step 4: Run the 4dvar!
Note this may take up to a minute to complete

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg4Dvar/yamls/4dvar_mult_obs.yaml

We can look at the increment throughout the window (we aren't plotting the observation locations for readability). 

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        --plotwind \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_inc_begin --title "Increment 00H: multiple obs"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        --plotwind \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_inc_mid --title "Increment 12H: multiple obs"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        --plotwind \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_inc_end --title "Increment 24H: multiple obs"


<center><img src="images/4dvar_inc_exp_mult.gif" width="700" /></center>


Unlike the single-observation experiments, the 00H increment now has structure across the whole domain. Each observation has contributed a piece of the picture, with its information propagated by the adjoint to $t_0$ and combined with the others.
Notice a few things:

- The increments are organized into coherent, meso-to-synoptic-scale features, not just isolated blobs around each ob location. 
- The pattern has a strong resemblance between Layer 1 and Layer 2, with broadly similar features in both layers but slightly different amplitudes and positions. This structure is partly a feature of the B-matrix (which couples the two layers) and partly a consequence of the dynamics.
- As the window evolves from 00H → 12H → 24H,  the increments deform and propagate with the background flow — just as in the single-ob experiments, but now with many features doing this simultaneously.

Here are some questions for interpreting these results:

1. Compare the 00H increment to the single-obs experiments. The pattern is much more complex...but can you still identify the key 4DVar behaviors (flow-following, focusing/deformation, advection) in the evolution from 00H → 24H?
2. The increments are organized into coherent features at scales much larger than any single observation. What two factors are responsible for spreading information from observation points into these broader patterns?

Let's also look at the analysis and forecast errors throughout the window:

In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P15D.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_an_begin --title "Analysis error - 00H (begin of window)"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P15DT12H.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_an_mid --title "Analysis error - 12H (mid of window)"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/da/4dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_an_end --title "Analysis error - 24H (end of window)"


And forecast error:

In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P15D.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_fc_begin --title "Background error - 00H (begin of window)"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P15DT12H.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_fc_mid --title "Background error - 12H (mid of window)"


In [ ]:
python ./plot.py qg fields \
        $JEDI_EDU/qg4Dvar/output/exp_mult/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --fieldmax 112662784 \
        --output $JEDI_EDU/qg4Dvar/output/exp_mult/plots/4dvar_fc_end --title "Background error - 24H (end of window)"


<center><img src="images/4dvar_mult_err.gif", width="700"/></center>

This experiment demonstrates several things:

4D-Var combines many observations coherently. Each ob contributes to the $t_0$
analysis through forward and backward (adjoint) propagation, weighted by the background and observation error covariances.
Information is spread by both statistics $\mathbf{B}$ and model dynamics $\mathbf{M}$. 

The assimilation provides a real reduction in error compared to the background — but the analysis is not perfect. Residual errors reflect the fundamental limits of any DA system: imperfect observations, imperfect error statistics, and imperfect models. Further tuning of the DA may yield better results, but these fundamental limits remain/

In an operational system, this analysis would be used as the initial condition for a forecast, and the cycle would repeat — with the new forecast becoming the next window's background. Over many cycles, the system converges toward a state that is consistently closer to the truth than any single forecast or any single observation could provide on its own.

## Summary

 **What 4D-Var does:** Finds the analysis state at $t_0$ such that, when propagated forward by the model, it best fits observations across the entire assimilation window — given the background and observation error statistics.

 **Key takeaways from each experiment:**

 - **Single obs at window start (exp_begin):** The 00H increment looks 3DVar-like (compact, B-matrix-shaped), then propagates forward with the flow at 12H and 24H — illustrating the **tangent linear model** in action.

 - **Single obs at window end (exp_end):** The 24H increment is the compact one, while the 00H increment is a broader, weaker pattern, the initial-condition correction the dynamics will sharpen into the obs-fitting feature. This illustrates the **adjoint** propagating sensitivity backward in time.

 - **Multiple obs throughout the window (exp_mult):** 4D-Var combines information from observations distributed in space *and* time, producing a coherent analysis with structure at scales larger than the obs spacing. The analysis produced has reduced error at all times throughout the DA window.

 - **Model dynamics incorporated in DA:** Across all experiments, increments aren't passively advected — they're advected, deformed, and amplified by the background flow. This is what gives 4DVar its flow-dependent, dynamically-consistent character that 3DVar fundamentally lacks.
